In [2]:
import pandas as pd
from datetime import datetime

def price_storage_contract(in_dates, out_dates, in_prices, out_prices, volume, max_volume, storage_cost_per_month, inj_with_cost_per_mmbtu, transport_cost_per_event):
    
    # Safety Check: Make sure we aren't trying to store more than the facility can hold
    if volume > max_volume:
        return "Error: Trade volume exceeds maximum storage capacity."
        
    total_contract_value = 0
    
    # We use a loop in case the client wants to do multiple trades in one contract
    for i in range(len(in_dates)):
        
        # 1. Calculate Gross Profit (Revenue from selling - Cost of buying)
        revenue = volume * out_prices[i]
        cost_to_buy = volume * in_prices[i]
        gross_profit = revenue - cost_to_buy
        
        # 2. Calculate how many months the gas is stored
        in_date = pd.to_datetime(in_dates[i])
        out_date = pd.to_datetime(out_dates[i])
        
        # Simple math to find the difference in months
        months_stored = (out_date.year - in_date.year) * 12 + (out_date.month - in_date.month)
        
        # 3. Calculate all the "Friction" Costs
        total_storage_cost = months_stored * storage_cost_per_month
        total_inj_with_cost = volume * inj_with_cost_per_mmbtu
        total_transport_cost = transport_cost_per_event * 2 # Multiply by 2 because we pay once to deliver, once to take away
        
        total_costs = total_storage_cost + total_inj_with_cost + total_transport_cost
        
        # 4. Final Math: Profit minus Costs
        net_trade_value = gross_profit - total_costs
        
        # Add this trade's value to the total contract
        total_contract_value += net_trade_value
        
    return total_contract_value

# --- Let's test it

# The text example: Buy 1 million MMBtu at $2, store for 4 months, sell at $3.
# Costs: $100K/month storage, $10K total inj/with cost, $50K transport each way.

test_in_dates = ['2024-06-01']  # Summer
test_out_dates = ['2024-10-01'] # 4 months later
test_in_prices = [2.0]
test_out_prices = [3.0]
test_volume = 1000000 # 1 Million MMBtu
test_max_volume = 2000000 # Plenty of room

test_storage_fee = 100000 
test_inj_with_fee = 0.01 # $10,000 divided by 1,000,000 MMBtu = $0.01 per MMBtu
test_transport_fee = 50000

final_value = price_storage_contract(
    test_in_dates, test_out_dates, 
    test_in_prices, test_out_prices, 
    test_volume, test_max_volume, 
    test_storage_fee, test_inj_with_fee, test_transport_fee)

print(f"The estimated value of this contract is: ${final_value:,.2f}")

The estimated value of this contract is: $490,000.00
